In [1]:
import os
import shutil
       
import json
import numpy as np

import vtkplotlib as vpl
from stl.mesh import Mesh

In [2]:
def visualize_stl(stl):
    mesh = Mesh.from_file(stl)
    
    # Plot the mesh, change th opacity to make the femur head point visible
    vpl.mesh_plot(mesh, opacity=1,color = 'white')
    
    # Plot the points
    # Change the background color
    vpl.gcf().background_color = (0,0,0)  # White background
    
    # vpl.plot(physical_point, color="blue", line_width=3)  # Blue line with thickness 3
    
    # Display the plot
    vpl.show()

In [ ]:
ap = "/Users/kamleshranabhat/Documents/naamii/PL19_00340_53/ankle_label_stl_file.stl"
kp = "/Users/kamleshranabhat/Documents/naamii/PL19_00340_53/knee_label_stl_file.stl"
pp = "/Users/kamleshranabhat/Documents/naamii/PL19_00340_53/pelvis_label_stl_file.stl"

visualize_stl(ap)
visualize_stl(kp)
visualize_stl(pp)

: 

In [ ]:
import trimesh
import open3d as o3d
import numpy as np

def trimesh_to_open3d(mesh):
    """
    Convert a Trimesh mesh to an Open3D TriangleMesh.

    Parameters:
    mesh (trimesh.Trimesh): Input mesh in Trimesh format.

    Returns:
    o3d.geometry.TriangleMesh: Converted Open3D mesh.
    """
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.faces)

    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
    o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
    o3d_mesh.compute_vertex_normals()  # Required for visualization and processing

    return o3d_mesh

def open3d_to_trimesh(o3d_mesh):
    """
    Convert an Open3D TriangleMesh back to a Trimesh mesh.

    Parameters:
    o3d_mesh (o3d.geometry.TriangleMesh): Input Open3D mesh.

    Returns:
    trimesh.Trimesh: Converted mesh in Trimesh format.
    """
    vertices = np.asarray(o3d_mesh.vertices)
    triangles = np.asarray(o3d_mesh.triangles)
    return trimesh.Trimesh(vertices=vertices, faces=triangles)

def smooth_stl_mesh(input_path, output_path, iterations=10):
    """
    Apply Laplacian smoothing to an STL file using Open3D.

    Parameters:
    input_path (str): Path to the input STL file.
    output_path (str): Path to save the smoothed STL file.
    iterations (int): Number of smoothing iterations (default is 10).
    """
    # Load the STL mesh using Trimesh
    mesh = trimesh.load(input_path, force='mesh')
    print(f"Loaded mesh with {len(mesh.vertices)} vertices and {len(mesh.faces)} faces")

    # Convert Trimesh mesh to Open3D format
    o3d_mesh = trimesh_to_open3d(mesh)

    # Apply Laplacian smoothing (reduces sharp edges while preserving shape)
    o3d_mesh = o3d_mesh.filter_smooth_laplacian(number_of_iterations=iterations)
    o3d_mesh.compute_vertex_normals()  # Recompute normals after smoothing

    # Convert the smoothed mesh back to Trimesh format
    smoothed_mesh = open3d_to_trimesh(o3d_mesh)

    # Export the smoothed mesh to a new STL file
    smoothed_mesh.export(output_path)
    print(f"Smoothed mesh saved as: {output_path}")

In [14]:
import trimesh
import numpy as np

def surface_nets_smoothing(mesh, iterations=1, influence=0.5):
    """
    Apply a simple Surface Nets-style smoothing to a Trimesh mesh.
    
    This algorithm moves each vertex towards the average of its neighbors,
    effectively reducing noise and smoothing the surface while maintaining
    structural features.

    Parameters:
    mesh (trimesh.Trimesh): The input mesh to smooth.
    iterations (int): Number of smoothing iterations to perform.
    influence (float): Weighting factor for smoothing (0-1).
                       Higher means more smoothing.

    Returns:
    trimesh.Trimesh: Smoothed mesh.
    """
    vertices = mesh.vertices.copy()
    adjacency = mesh.vertex_neighbors  # List of neighboring vertices for each vertex

    for _ in range(iterations):
        new_vertices = vertices.copy()

        for i, neighbors in enumerate(adjacency):
            if len(neighbors) < 3:
                continue  # Skip poorly connected vertices

            # Compute average position of neighboring vertices
            neighbor_positions = vertices[neighbors]
            avg_position = neighbor_positions.mean(axis=0)

            # Smooth vertex by moving toward the average neighbor position
            new_vertices[i] = vertices[i] * (1 - influence) + avg_position * influence

        vertices = new_vertices  # Update for next iteration

    return trimesh.Trimesh(vertices=vertices, faces=mesh.faces)

def apply_surface_nets(stl_path, surfacenets_output_mesh_path, iterations=10, influence=0.4):
    """
    Apply surface nets smoothing to an STL mesh and save the result.

    Parameters:
    stl_path (str): Path to the input STL file.
    surfacenets_output_mesh_path (str): Path to save the smoothed STL.
    iterations (int): Number of smoothing iterations.
    influence (float): Weighting factor for smoothing.
    """
    # Load mesh
    original_mesh = trimesh.load_mesh(stl_path)

    # Apply smoothing
    smoothed_mesh = surface_nets_smoothing(original_mesh, iterations=iterations, influence=influence)

    # Save the result
    smoothed_mesh.export(surfacenets_output_mesh_path)
    print("Successfully applied surface nets smoothing.")

In [ ]:
input_mesh_path = "/Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/PL_307/pelvis/pelvis_stl_file.stl"
laplacian_output_mesh_path = "/Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/PL_307/pelvis/laplacian_smoothing_pelvis_stl_file.stl"
surfacenets_output_mesh_path = "/Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/PL_307/pelvis/surfacenets_smoothing_pelvis_stl_file.stl"
smooth_stl_mesh(input_mesh_path, laplacian_output_mesh_path, iterations=20)
apply_surface_nets(input_mesh_path, surfacenets_output_mesh_path, iterations=10, influence=0.6)


In [ ]:
visualize_stl(input_mesh_path)
visualize_stl(laplacian_output_mesh_path)
visualize_stl(surfacenets_output_mesh_path)

In [ ]:
from pathlib import Path
import os
import vtk
import SimpleITK as sitk
import numpy as np
 
def save_stl_file(segmentation_mask_path: Path, organ: str, task_dir: str):
    try:
        path = str(segmentation_mask_path)
 
        # --- Step 1: Read image metadata using SimpleITK ---
        itk_image = sitk.ReadImage(path)
        spacing = itk_image.GetSpacing()
        origin = itk_image.GetOrigin()
        direction = itk_image.GetDirection()
        direction_matrix = np.array(direction).reshape(3, 3)
 
        # --- Step 2: Read the NIfTI image using VTK ---
        reader = vtk.vtkNIFTIImageReader()
        reader.SetFileName(path)
        reader.Update()
 
        # --- Step 3: Extract surface using Discrete Marching Cubes ---
        contour = vtk.vtkDiscreteMarchingCubes()
        contour.SetInputConnection(reader.GetOutputPort())
        contour.SetValue(0, 1)  # label to extract
        contour.Update()
 
        # --- Step 4: Apply direction + origin transformation ---
        vtk_matrix = vtk.vtkMatrix4x4()
        for i in range(3):
            for j in range(3):
                vtk_matrix.SetElement(i, j, direction_matrix[i, j])
            vtk_matrix.SetElement(i, 3, origin[i])
        vtk_matrix.SetElement(3, 0, 0)
        vtk_matrix.SetElement(3, 1, 0)
        vtk_matrix.SetElement(3, 2, 0)
        vtk_matrix.SetElement(3, 3, 1)
 
        transform = vtk.vtkTransform()
        transform.SetMatrix(vtk_matrix)
 
        transform_filter = vtk.vtkTransformPolyDataFilter()
        transform_filter.SetTransform(transform)
        transform_filter.SetInputConnection(contour.GetOutputPort())
        transform_filter.Update()
 
        # # --- Step 5: Smooth the mesh (optional) ---
        smoother = vtk.vtkWindowedSincPolyDataFilter()
        smoother.SetInputConnection(transform_filter.GetOutputPort())
        smoother.SetNumberOfIterations(20)
        smoother.BoundarySmoothingOff()
        smoother.FeatureEdgeSmoothingOff()
        smoother.SetFeatureAngle(10.0)
        smoother.SetPassBand(0.0001)
        smoother.NonManifoldSmoothingOn()
        smoother.NormalizeCoordinatesOn()
        smoother.Update()
 
        # --- Step 6: Write STL to file ---
        os.makedirs(task_dir, exist_ok=True)
        output_path = os.path.join(task_dir, f"{organ}_stl_file.stl")
 
        writer = vtk.vtkSTLWriter()
        writer.SetFileName(output_path)
        writer.SetInputConnection(transform_filter.GetOutputPort())
        writer.Write()
 
        print(f"STL file saved to: {output_path}")
 
    except Exception as e:
        print(f"Error occurred: {e}")
 

In [1]:
from pathlib import Path
import os
import vtk
import SimpleITK as sitk
import numpy as np
import trimesh

def vtk_polydata_to_trimesh(polydata: vtk.vtkPolyData):
    # Get vertices
    points = polydata.GetPoints()
    vertices = np.array([points.GetPoint(i) for i in range(points.GetNumberOfPoints())])

    # Get faces
    faces = []
    for i in range(polydata.GetNumberOfCells()):
        cell = polydata.GetCell(i)
        if cell.GetNumberOfPoints() == 3:  # only triangles
            faces.append([cell.GetPointId(j) for j in range(3)])
    faces = np.array(faces)

    return trimesh.Trimesh(vertices=vertices, faces=faces)

def save_stl_file(segmentation_mask_path: Path, organ: str, task_dir: str):
    try:
        path = str(segmentation_mask_path)

        # --- Step 1: Read image metadata using SimpleITK ---
        itk_image = sitk.ReadImage(path)
        spacing = itk_image.GetSpacing()
        origin = itk_image.GetOrigin()
        direction = itk_image.GetDirection()
        direction_matrix = np.array(direction).reshape(3, 3)

        # --- Step 2: Read the NIfTI image using VTK ---
        reader = vtk.vtkNIFTIImageReader()
        reader.SetFileName(path)
        reader.Update()

        # --- Step 3: Extract surface using Discrete Marching Cubes ---
        contour = vtk.vtkDiscreteMarchingCubes()
        contour.SetInputConnection(reader.GetOutputPort())
        contour.SetValue(0, 1)  # label to extract
        contour.Update()

        # --- Step 4: Apply direction + origin transformation ---
        vtk_matrix = vtk.vtkMatrix4x4()
        for i in range(3):
            for j in range(3):
                vtk_matrix.SetElement(i, j, direction_matrix[i, j])
            vtk_matrix.SetElement(i, 3, origin[i])
        vtk_matrix.SetElement(3, 0, 0)
        vtk_matrix.SetElement(3, 1, 0)
        vtk_matrix.SetElement(3, 2, 0)
        vtk_matrix.SetElement(3, 3, 1)

        transform = vtk.vtkTransform()
        transform.SetMatrix(vtk_matrix)

        transform_filter = vtk.vtkTransformPolyDataFilter()
        transform_filter.SetTransform(transform)
        transform_filter.SetInputConnection(contour.GetOutputPort())
        transform_filter.Update()

        # --- Step 5: Get vtkPolyData from transform filter
        polydata = transform_filter.GetOutput()

        # --- Step 6: Apply Surface Nets-style smoothing using trimesh ---
        # Convert to trimesh
        mesh = vtk_polydata_to_trimesh(polydata)

        # Apply surface nets smoothing (same as before)
        vertices = mesh.vertices.copy()
        faces = mesh.faces
        adjacency = mesh.vertex_neighbors

        iterations = 40
        influence = 0.6

        for _ in range(iterations):
            new_vertices = vertices.copy()
            for i, neighbors in enumerate(adjacency):
                if len(neighbors) < 3:
                    continue
                avg_position = vertices[neighbors].mean(axis=0)
                new_vertices[i] = vertices[i] * (1 - influence) + avg_position * influence
            vertices = new_vertices

        smoothed_mesh = trimesh.Trimesh(vertices=vertices, faces=faces)

        # Final STL output path
        output_path = os.path.join(task_dir, f"{organ}_stl_file.stl")
        smoothed_mesh.export(output_path)
        print(f"Smoothed STL file saved to: {output_path}")

    except Exception as e:
        print(f"Error occurred: {e}")

In [2]:
# Example usage
mask_path = "/Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/3806-2024-10-28-03-07-26/knee/knee_label.nii.gz"
task_dir = '/Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/PL_307/knee'
organ = 'test'
 
save_stl_file(mask_path, organ, task_dir)

Smoothed STL file saved to: /Users/kamleshranabhat/Documents/naamii/mask_to_stl/OneDrive_1_5-8-2025/PL_307/knee/test_stl_file.stl


In [4]:
import os
import numpy as np
import SimpleITK as sitk
from vedo import Volume, write as vedo_write

In [17]:
import os
import numpy as np
import SimpleITK as sitk
from vedo import Volume, write as vedo_write
import trimesh


def resample_image(itk_image: sitk.Image, out_spacing=(1.0, 1.0, 1.0)):
    """
    Resample itk_image to new out_spacing
    """
    original_spacing = itk_image.GetSpacing()
    original_size = itk_image.GetSize()
    out_size = [int(round(osz * ospc / nspc)) for osz, ospc, nspc in zip(original_size, original_spacing, out_spacing)]

    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(out_spacing)
    resample.SetSize(out_size)
    resample.SetOutputDirection(itk_image.GetDirection())
    resample.SetOutputOrigin(itk_image.GetOrigin())
    resample.SetTransform(sitk.Transform())
    resample.SetDefaultPixelValue(itk_image.GetPixelIDValue())
    resample.SetInterpolator(sitk.sitkNearestNeighbor)
    return resample.Execute(itk_image)


def save_stl_file(segmentation_mask: sitk.Image, organ: str, task_dir: str,
                  smooth_iterations=10, smooth_influence=0.4):
    """
    Save STL file from segmentation mask, applying vedo mesh generation + surface nets smoothing.
    """
    try:
        # Resample image and extract mask array
        new_mask_img = resample_image(segmentation_mask)
        new_mask_arr = sitk.GetArrayFromImage(new_mask_img)
        new_mask_arr[0] = 0
        new_mask_arr[-1] = 0

        # Create vedo volume and isosurface
        vol = Volume(np.transpose(new_mask_arr, axes=[2, 1, 0]),
                     spacing=new_mask_img.GetSpacing(),
                     origin=new_mask_img.GetOrigin())
        m0 = vol.isosurface(flying_edges=True)
        m1 = m0.clone().smooth(niter=50, boundary=True)

        # Convert vedo mesh (vedo.Mesh) to trimesh.Trimesh
        vertices = np.array(m1.points)
        faces = np.array(m1.faces())

        raw_trimesh = trimesh.Trimesh(vertices=vertices, faces=faces)

        # Apply surface nets smoothing directly
        vertices = raw_trimesh.vertices.copy()
        adjacency = raw_trimesh.vertex_neighbors

        for _ in range(smooth_iterations):
            new_vertices = vertices.copy()

            for i, neighbors in enumerate(adjacency):
                if len(neighbors) < 3:
                    continue
                neighbor_positions = vertices[neighbors]
                avg_position = neighbor_positions.mean(axis=0)
                new_vertices[i] = vertices[i] * (1 - smooth_influence) + avg_position * smooth_influence

            vertices = new_vertices

        smoothed_mesh = trimesh.Trimesh(vertices=vertices, faces=raw_trimesh.faces)

        # Save final STL
        final_stl_path = os.path.join(task_dir, f"{organ}_stl_file.stl")
        smoothed_mesh.export(final_stl_path)

        print(f"Successfully saved and smoothed STL for {organ}: {final_stl_path}")
    except Exception as e:
        print(f"Error encountered while saving STL file for {organ}.")
        raise e

In [18]:
# Example usage
mask_path = "/Users/kamleshranabhat/Documents/naamii/unilateral_knee_2/knee_label.nii.gz"
task_dir = '/Users/kamleshranabhat/Documents/naamii/unilateral_knee_2'
organ = 'test'
segmentation_mask = sitk.ReadImage(mask_path)
save_stl_file(segmentation_mask, organ, task_dir, 15, 0.6)   

Successfully saved and smoothed STL for test: /Users/kamleshranabhat/Documents/naamii/unilateral_knee_2/test_stl_file.stl


In [ ]:
input_mesh_path = "/Users/kamleshranabhat/Documents/naamii/unilateral_knee_2/test_stl_file.stl"
visualize_stl(input_mesh_path)

In [ ]:
import nibabel as nib
import numpy as np

def merge_nifti_to_binary(nifti_path1, nifti_path2, output_path):
    # Load the two images
    img1 = nib.load(nifti_path1)
    img2 = nib.load(nifti_path2)

    # Extract data as numpy arrays
    data1 = img1.get_fdata()
    data2 = img2.get_fdata()

    # Create binary merged data: 1 if either image has a non-zero voxel, else 0
    merged_data = np.logical_or(data1 > 0, data2 > 0).astype(np.uint8)

    # Use the affine and header from img1
    merged_img = nib.Nifti1Image(merged_data, affine=img1.affine, header=img1.header)

    # Save to output path in .nii.gz format
    nib.save(merged_img, output_path)
    print(f"Saved merged binary image to: {output_path}")

In [ ]:
n = "/Users/kamleshranabhat/Documents/naamii/tibia.nii"
m = "/Users/kamleshranabhat/Documents/naamii/femur.nii"
s = "/Users/kamleshranabhat/Documents/naamii/knee_label.nii.gz"
merge_nifti_to_binary(n, m, s)

In [1]:
import SimpleITK as sitk
import numpy as np
# m = "/Users/kamleshranabhat/Documents/naamii/pelvis_label.nii.gz"
mp = "/Users/kamleshranabhat/Documents/naamii/sorted_unilateral_ankles_kamlesh/PL20_01088_42/ankle_1/Segmentation.nii"
p = "/Users/kamleshranabhat/Documents/naamii/sorted_unilateral_ankles_kamlesh/PL20_01088_42/ankle_1/ankle_label.nii.gz"

def transform(mask_path, save_path):
    mask_image = sitk.ReadImage(mask_path)
    mask_array = sitk.GetArrayFromImage(mask_image)
    rotated_mask = np.rot90(mask_array, k=2)
    flipped_mask = np.fliplr(rotated_mask)
    mask = sitk.GetImageFromArray(flipped_mask)
    sitk.WriteImage(mask, save_path)

transform(mp, p)